## Agentic AI Analytics Bot v1.6.0

Features:
- 5-stage analytics pipeline: classify → plan → execute → narrate → guardrails
- LLM-based mode selection (Retrieve / Explore / Reason)
- Dual mode control: auto (LLM picks per interaction) or manual (slash commands lock mode)
- Slash commands: /retrieve /explore /reason (lock) · /auto (unlock)
- Three-tier model routing: Haiku/8B-instant (fast), Sonnet/Llama-70B (default), Opus 4.7 / GPT-OSS-120B (Reason narration)
- Rolling session summary: bounded-context follow-up resolution across long sessions
- Automatic schema-aware query generation (DuckDB)
- DOC_LOOKUP: answers metadata questions directly from data model documentation
- guard_sql: AST-level SELECT-only enforcement via sqlglot (blocks DDL, DML, system tables)
- SQL error retry: targeted per-query fix prompt (SQL_RETRY_PROMPT), not full re-plan
- verify_groundedness: regex number/entity extraction vs. query results (no LLM call)
- check_compliance: tier-0 LLM scan for 5 violation categories (approximation, evaluative, inference, unit drift, unsupported claims)
- Soft-block: ⚠ caveat appended to narrative when guardrail issues detected
- Injection-based guardrail eval harness: 18 cases across SQL / narrative / prompt injection stages
- Cross-model grading: assessor_llm_fn param in run_eval() and run_guardrail_eval()
- Centralized prompt management (prompts.py)

Validation: 50/50 standard suite (Mode 0) · guardrail injection suite pending full run


## Quick Start
1. Run Blocks 1–5 in order (one-time setup)
2. **Chat mode:** Run Block 6 — type questions, type `quit` to exit
3. **Validation:** In Block 7, uncomment `results = run_eval(...)` and run
4. **Ad-hoc SQL:** After Block 3, use `run_query("SELECT ...")` in any cell
5. **Iterate on prompts:** Edit `prompts.py`, then rerun Block 5, 6 or 7 (modules auto-reload)

In [3]:
# ============================================================
# Block 1: Environment setup
# ============================================================

# ── Project structure ───────────────────────────────────────
PROJECT_ROOT_NAME       = "analytics_agent"
PROJECTS_PARENT_DIR     = "/content/drive/MyDrive/Projects"

CODE_MODULES_SUBDIR     = "code_modules"
DATA_SCHEMA_SUBDIR      = "data_schema"
LOGS_SUBDIR             = "logs"

SCHEMA_FILE_NAME        = "data_model.json"

# ── API Secret names ────────────────────────────────────────
# must match what you set in Colab Secrets
API_SECRET_NAMES = {
    "claude": "Claude",
    "groq":   "Groq",
    "gemini": "PersonalAPIKey",
}

print("All ok")

All ok


In [4]:
# ============================================================
# Block 2: Master initialization
# ============================================================

import importlib, subprocess, sys, shutil
from pathlib import Path

# ── Install packages ────────────────────────────────────────
INSTALL_PACKAGES = ["anthropic", "groq", "duckdb", "pydrive2"]

def ensure_packages(packages):
    for pkg in packages:
        try:
            importlib.import_module(pkg)
        except ImportError:
            print(f"Installing: {pkg}")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

ensure_packages(INSTALL_PACKAGES)
print("Packages OK")

# ── Mount Google Drive ──────────────────────────────────────
def ensure_drive_mounted(mount_point="/content/drive"):
    mount_point = Path(mount_point)
    if (mount_point / "MyDrive").exists():
        return
    from google.colab import drive
    drive.mount(str(mount_point))

ensure_drive_mounted()
print("Drive OK")

# ── Resolve paths ────────────────────────────────────────────
PROJECT_ROOT     = (Path(PROJECTS_PARENT_DIR) / PROJECT_ROOT_NAME).resolve()
CODE_MODULES_DIR = (PROJECT_ROOT / CODE_MODULES_SUBDIR).resolve()
DATA_SCHEMA_DIR  = (PROJECT_ROOT / DATA_SCHEMA_SUBDIR).resolve()
LOGS_DIR         = (PROJECT_ROOT / LOGS_SUBDIR).resolve()
SCHEMA_PATH      = (DATA_SCHEMA_DIR / SCHEMA_FILE_NAME).resolve()

# ── Register code_modules on sys.path ────────────────────────
for d in [CODE_MODULES_DIR] + [CODE_MODULES_DIR / sub for sub in ["_data", "_skills", "_agent"]]:
    shutil.rmtree(str(d / "__pycache__"), ignore_errors=True)

if str(CODE_MODULES_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_MODULES_DIR))

print(f"Path: {CODE_MODULES_DIR}")
print("Init OK")

# ── Define a function to reload all code modules ────────────
def reload_code_modules():
    """Reload all project modules from GDrive after editing source files."""
    import types
    prefix = str(CODE_MODULES_DIR)
    reloaded = []
    for name, mod in list(sys.modules.items()):
        if isinstance(mod, types.ModuleType) and hasattr(mod, '__file__') and mod.__file__ and mod.__file__.startswith(prefix):
            importlib.reload(mod)
            reloaded.append(name)
    if reloaded:
        print(f"Reloaded {len(reloaded)} modules")

Packages OK
Mounted at /content/drive
Drive OK
Path: /content/drive/MyDrive/Projects/analytics_agent/code_modules
Init OK


In [5]:
# ============================================================
# Block 3: Backend config
# ============================================================

from _skills import llm_backends
reload_code_modules()

from _skills.llm_backends import init_backends, call_llm, MODEL_MAP

BACKEND = "claude"                       # "claude", "groq", or "gemini"
BACKENDS_TO_INIT = [BACKEND]             # init only what you need

result = init_backends(BACKENDS_TO_INIT, secret_names=API_SECRET_NAMES, test=True)
llm_clients = result["clients"]

for name, reply in result["pings"].items():
    print(f"  {name:<7} ✓  {reply}")

def llm(prompt, tier=1):
    return call_llm(prompt, backend=BACKEND, clients=llm_clients, model=tier)


Reloaded 2 modules
  claude  ✓  "Hi there, friend!" 👋


In [6]:
# ============================================================
# Block 4: Data — schema, datasets, DuckDB
# ============================================================

from _data import olist_schema_and_datasets
from _skills import duckdb_utils
reload_code_modules()

from _data.olist_schema_and_datasets import init_all
from _skills.duckdb_utils import run_query as duckdb_run_query, validate_tables

result = init_all(schema_path=SCHEMA_PATH, validate_fn=validate_tables)

data_model    = result["data_model"]
SCHEMA        = result["SCHEMA"]
tables_meta   = result["tables_meta"]
datasets      = result["datasets"]
duckdb_tables = result["duckdb_tables"]
tables        = duckdb_tables  # alias for agent functions

# ── Convenience: ad-hoc SQL from notebook ───────────────────
def run_query(sql):
    return duckdb_run_query(sql=sql, tables=tables)

Reloaded 5 modules
Schema loaded: 7 tables, 6,450 chars

  orders           99,441 rows  cols=['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
  products         32,951 rows  cols=['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
  reviews          99,224 rows  cols=['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
  customers        99,441 rows  cols=['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
  order_items     112,650 rows  cols=['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
  payments        10

In [7]:
# ============================================================
# Block 5: Agent + Chat setup
# ============================================================

from _agent import (
    analyst_agent as _aa_mod,
    chat as _chat_mod,
    prompts as _prompts_mod,
    conversation_processor as _cp_mod,   # NEW
)
from _skills import session_logger
reload_code_modules()

from _agent.analyst_agent import analyst_agent as _analyst_agent, MODE_NAMES
from _agent.chat import create_chat_fn
from _skills.duckdb_utils import execute_queries

def agent(question, mode=0, user_context=None):
    return _analyst_agent(
        question=question, schema=SCHEMA, tables=tables,
        llm_fn=llm, execute_fn=execute_queries,
        mode=mode, user_context=user_context,
    )

chat = create_chat_fn(
    agent_fn=agent,
    llm_fn=llm,
    logs_dir=LOGS_DIR,
    backend=BACKEND,
    model_name=MODEL_MAP[BACKEND][1],
)

print("Agent is ready")


Reloaded 11 modules
Agent is ready


In [8]:
# ============================================================
# Block 6: Interactive Chat / Analytics Agent
# ============================================================

chat()

Session log: /content/drive/MyDrive/Projects/analytics_agent/logs/chat_log_2026_04_28_16_02.md
Analytics Agent [claude]  •  Retrieve mode (auto)
Commands: /retrieve /explore /reason (lock mode) · /auto (unlock) · quit

You: Describe available data
  → Interpreting...
  → Interaction type: standalone
  → Mode: Retrieve
  → Planning queries...


**Bot:** ## Available Data: Olist Brazilian E-commerce Data Product

### Tables Overview

| Table Name | Description | Key KPIs | Key Business Dimensions | Row Count |
|---|---|---|---|---|
| orders | Central fact table; one row per order across the full lifecycle | Order volume, status distribution, delivery timeliness | Order status, purchase date, delivery dates | ~100,000 |
| order_items | Item-level detail; one row per item within an order | Revenue (price + freight), item counts, freight costs | Product, seller, shipping deadline | ~100,000+ |
| products | Product dimension; one row per product | Category revenue/volume, product attributes | Category (Portuguese), weight, dimensions, photos | Varies |
| customers | Customer dimension; one row per order-scoped customer ID | Distinct customer counts (via unique ID), regional demand | City, state, zip code | ~100,000 |
| sellers | Seller dimension; one row per seller | Seller counts, geographic distribution | City, state, zip code | Varies |
| payments | Payment fact; one row per installment/payment method | Total payment value, installment behavior, method mix | Payment type, installment count | ~100,000+ |
| reviews | Review fact; one row per review | Average satisfaction score, response time | Review score (1–5), review date | ~100,000 |

---

### How the Data Connects

Orders is the central hub of the data: each order links outward to the customer who placed it (for location and identity), to the individual items within it (for product, seller, price, and freight detail), to the payments made against it (for method and installment behavior), and to the review submitted after delivery (for satisfaction scoring). Products and sellers are reference dimensions that enrich the item-level data with category attributes and seller geography respectively.

---

### Valid Use Cases
- Order volume and revenue trends over time (2016–2018)
- Seller performance and geographic distribution
- Payment method and installment behavior analysis
- Regional/geographic demand by customer city and state
- Delivery timeliness (on-time vs. late vs. estimated)
- Product category popularity and freight cost analysis
- Customer satisfaction scoring by category, seller, or region

### Invalid Use Cases
- Real-time or current inventory
- True customer identity resolution (IDs are anonymized)
- Profitability or margin analysis (no cost data available)
- Returns or refunds tracking
- Budget or target comparisons
- Marketing attribution
- Any data outside the 2016–2018 window

---

**Caveats:** Monetary values are in Brazilian Reais (BRL). `customer_id` is order-scoped and must not be used for distinct customer counts — use `customer_unique_id` instead. Product category names are in Portuguese. Payment values may differ slightly from order item totals due to vouchers or rounding.

  → Updating summary...
⏱  24.94s total



You: How many customers do we have?
  → Interpreting...
  → Interaction type: standalone
  → Mode: Retrieve
  → Planning queries...


**Bot:** Across the full 2016–2018 dataset, Olist served **96,096** distinct customers, deduplicated at the true unique-customer level to eliminate order-scoped duplicates.

...

**Bot:** The question is unclear. The user is asking for a **subjective performance judgment** ("how good would you say this performance is?") based on a set of KPIs they've already computed. This is an evaluative/benchmarking question, not an empirical data extraction question. Answering it meaningfully requires one of the following: (a) an external benchmark or industry reference (e.g., average e-commerce on-time delivery rates in Brazil, typical marketplace cancellation rates, average review scores for comparable platforms) — none of which exist in this data model; (b) an internal benchmark defined by the user (e.g., a target, a prior period baseline, a peer comparison); or (c) a reframing into an empirically decomposable question the data *can* answer, such as identifying which segments are dragging or driving the aggregate KPIs. The data model contains no cost data, no targets, no external benchmarks, and no competitor data — so a grounded performance verdict cannot be produced from the data alone. The question as posed requires an analyst opinion or external context, not a query.

**What is specifically ambiguous:** The question asks for a normative judgment ("how good") without specifying the comparison basis — against what standard should 92.1% on-time delivery, 4.017/5.0 review score, or 0.63% cancellation rate be evaluated?

---

**Suggested reformulations — pick one:**

**Option 1 — Internal trend benchmark (data can fully answer):**
> "How did these KPIs (on-time delivery rate, average review score, cancellation rate, net revenue) evolve month-over-month across 2016–2018? Were they improving or deteriorating by end of period?"
→ This lets the data reveal directional health without needing external benchmarks.

**Option 2 — Segment decomposition (data can fully answer):**
> "Which states, product categories, or seller segments are underperforming on delivery timeliness and review scores — and how much do they drag on the aggregate metrics?"
→ This surfaces where the business has risk pockets beneath the headline numbers.

**Option 3 — Stress indicator analysis (data can fully answer):**
> "During the high-growth periods in 2017–2018, did delivery timeliness and review scores deteriorate — suggesting the operation was being stressed by volume?"
→ Tests whether the growth was sustainable or came at a quality cost.

Please confirm which framing fits your decision need, or specify an external benchmark you'd like to use as the reference point.

  → Updating summary...
⏱  62.75s total

You: /auto Go with 1
  → Interpreting...
  → Resolved as: How did these KPIs (on-time delivery rate, average review score, cancellation rate, net revenue) evolve month-over-month across 2016–2018? Were they improving or deteriorating by end of period?
  → Interaction type: continuation
  → Mode: Switching to Explore mode, for more comprehensive answers.
  → Planning queries...


**Bot:** ## KPI Evolution: Month-over-Month Trends, 2016–2018

**Analytical framing:** All four KPIs are measured at calendar-month grain using order purchase timestamp. On-time delivery rate covers delivered orders with both actual and estimated delivery dates present. Cancellation rate uses all orders placed that month (transitional statuses included in denominator per the planning assumptions). Net revenue excludes freight and canceled orders. 2016 is treated as partial/sparse (329 total orders across three months); trend conclusions focus on 2017–2018.

---

### 1. Net Revenue — Strong Growth, Sustained Through 2018

Revenue scaled dramatically from 2017 to 2018:

- **2017 full year:** BRL 6,108,492.27 across 45,101 orders
- **2018 full year (Jan–Aug, dataset end):** BRL 7,341,182.41 across 54,011 orders — already exceeding 2017 on fewer months of data

Monthly progression shows consistent growth: from BRL 120,098.27 in January 2017, revenue climbed to BRL 1,003,862.14 in November 2017 (Black Friday effect, 7,544 orders — the single highest-volume month in 2017). In 2018, monthly revenue ranged from BRL 837,895.43 (February) to BRL 993,592.98 (April), with June 2018 at BRL 863,265.53.

**Trajectory by end of period:** Revenue was still elevated in 2018 vs. 2017 on a comparable-month basis, though individual months showed volatility. The dataset cuts off after August 2018 for meaningful data.

---

### 2. On-Time Delivery Rate — Deteriorating Into 2018, with Recovery in Mid-2018

This is the most visible negative trend in the dataset:

| Period | On-Time Rate |
|---|---|
| 2016 (annual) | 98.50% |
| 2017 (annual) | 93.37% |
| 2018 (annual, Jan–Aug) | 90.63% |

Monthly detail reveals the deterioration was not linear:

- **2017:** Rates were relatively stable in the 94–97% range (May–August: 96.39%, 96.14%, 96.57%, 96.68%), then dropped sharply to **85.69% in November 2017** — coinciding with the volume spike (7,544 orders). December 2017 partially recovered to 91.62%.
- **2018:** February dropped to **84.01%** and March to a period low of **78.64%** (1,496 late orders out of 7,003 delivered). April recovered sharply to 94.69%, May to 91.76%, and June reached **98.64%** — the best month since early 2016.
- **July–August 2018:** Pulled back again to 95.52% and 89.61%.

The late-delivery decomposition adds nuance: average days-vs-estimate (negative = early) narrowed from –27.4 days in January 2017 to –6.4 days in March 2018, meaning the buffer between actual and estimated delivery compressed significantly. By August 2018, average actual delivery days had fallen to 7.7 days (vs. 19.6 days in October 2016), suggesting faster deliveries overall — but the tighter estimates left less margin for error.

**Trajectory by end of period:** On-time rate was deteriorating year-over-year (93.37% → 90.63%) with a mid-2018 recovery in June that did not hold into August.

---

### 3. Average Review Score — Stable Around 4.0–4.1, with a 2018 Dip and Late Recovery

- **2016 (annual):** 3.543 (sparse data, low reliability)
- **2017 (annual):** 4.087
- **2018 (annual):** 4.089

Monthly scores were remarkably stable across 2017 (range: 3.911 in November 2017 to 4.236 in August 2017). The November 2017 dip to 3.911 aligns with the same volume spike and on-time delivery degradation.

In 2018, scores dipped: **3.829 in February** and **3.752 in March** — the two months with the worst on-time delivery rates (84.01% and 78.64%). Scores then recovered: June 2018 reached **4.278** (highest in the full dataset), July 4.264, August 4.256.

**Trajectory by end of period:** Annual averages were flat (4.087 vs. 4.089), masking a Q1 2018 dip and Q2–Q3 2018 recovery. The final months in the dataset show the highest review scores of the entire period.

---

### 4. Cancellation Rate — Low and Stable in 2017–2018, with Outliers

- **2016 (annual):** 7.90% — heavily distorted by September 2016 (50% cancellation rate on just 4 orders)
- **2017 (annual):** 0.59%
- **2018 (annual):** 0.62%

Monthly rates in 2017 ranged from 0.19% (December) to 1.23% (March). In 2018, the range was 0.22% (April) to 1.29% (August). No sustained upward or downward trend is visible — the metric oscillated within a narrow 0.2–1.3% band across both full years.

September and October 2018 show 93.75% and 100.00% cancellation rates respectively, but with only 16 and 4 orders — these are dataset tail artifacts, not operational signals.

**Trajectory by end of period:** Flat and low. No deterioration or improvement signal in 2017–2018.

---

### Summary: Improving or Deteriorating?

| KPI | 2017→2018 Direction | End-of-Period Signal |
|---|---|---|
| Net Revenue | ↑ Improving | Elevated; growth sustained |
| On-Time Delivery Rate | ↓ Deteriorating (YoY) | Mid-2018 recovery did not hold into August |
| Average Review Score | → Flat (4.087 → 4.089) | Recovered to period highs in June–August 2018 |
| Cancellation Rate | → Flat (0.59% → 0.62%) | No directional trend |

---

### Notable Findings

- **November 2017 stress event:** The single largest order month (7,544 orders) drove the on-time rate to 85.69%, review scores to their 2017 low (3.911), and generated BRL 1,003,862.14 in revenue — all three KPIs moved together.
- **Feb–March 2018 delivery degradation:** On-time rates of 84.01% and 78.64% represent the worst two consecutive months in the dataset. Average days late vs. estimate narrowed to –8.3 and –6.4 days respectively, vs. the 2017 norm of –11 to –13 days.
- **June 2018 anomaly:** On-time delivery rate of 98.64% and review score of 4.278 were both period highs despite 6,167 orders — a notable positive outlier in an otherwise declining on-time trend.
- **Delivery speed vs. reliability tradeoff:** Actual delivery times shortened from ~19.6 days (Oct 2016) to 7.7 days (Aug 2018), but on-time rates declined — indicating estimated delivery dates tightened faster than actual delivery times improved.

---

**Caveats:**
- 2016 data covers only September, October, and December with a combined 329 orders; annual 2016 figures are not comparable to 2017–2018.
- September and October 2018 have 16 and 4 orders respectively and should be excluded from trend analysis.
- Review counts slightly exceed order counts in some months due to join behavior; the discrepancy is small (< 10 rows/month) and does not materially affect averages.
- The dataset ends in August 2018 for operational purposes; no Q4 2018 comparisons are possible.

---

Would you like to break down on-time delivery rate by customer state or region next to identify whether the Feb–March 2018 degradation was geographically concentrated?

  → Updating summary...
⏱  130.91s total



You: quit
Session ended.


In [ ]:
# ============================================================
# Block 7: Validation (optional)
# ============================================================

from _agent import prompts as _prompts_mod, analyst_agent as _aa_mod
from _skills import eval_runner
from _data import olist_test_cases
reload_code_modules()

from _skills.eval_runner import run_eval
from _data.olist_test_cases import discovery, easy, hard, misleading, multistage, realistic

from IPython.display import display, Markdown, HTML

# ── Custom one-off test questions ───────────────────────────

## Question (activate when needed)
q = "How many customers do we have?"

## Single-mode (activate when needed)
# r = agent(q, mode=0)
# print("=== Stage 1 (Classification) ===")
# print(r["stage1"])
# print("\n=== Execution Plan ===")
# print(r["execution_plan"])
# print("\n=== Raw Answer ===")
# print(r["answer"])
# print("\n=== Narrative ===")
# display(Markdown(f"**Q:** {q}\n\n{r['narrative']}"))

## Compare modes (activate when needed)
# for m in [0, 1, 2]:
#     r = agent(q, mode=m)
#     print(f"\n=== Mode {m} ===")
#     display(Markdown(f"**Q:** {q}\n\n{r['narrative']}"))


# ── Structured evaluation based on cases file ───────────────

## Pick which sets to run — comment/uncomment as needed
validation = {
    "Metadata / Descriptive": discovery,
    "Easy": easy,
    "Hard": hard,
    "Misleading": misleading,
    "Multi-stage": multistage,
    "Realistic": realistic,
}

## Run selected evaluations, log results
results = run_eval(agent, llm, validation, logs_dir=LOGS_DIR, mode=0)

Reloaded 13 modules

  Model Validation: Metadata / Descriptive, Easy, Hard, Misleading, Multi-stage, Realistic
  50 questions across 6 set(s)

  Metadata / Descriptive (6 questions)

  Q1: Describe the data you have access to.
  Narrative: ## Data Overview

| Table Name | Description | Key KPIs | Key Business Dimensions | Row Count |
|---|---|---|---|---|
| orders | Central fact table; one row per order covering the full order lifecycle | Order count, order status distribution, delivery timeliness, processing lag | Order status, purch...
  Assessment: The bot excellently covered all 7 tables with descriptions, KPIs, dimensions, and a clear conceptual explanation of orders as the central hub, and added valuable context through valid/invalid use cases. The only minor gap is that row counts were approximated or listed as "Varies" for several tables rather than providing specific figures, which falls slightly short of the precision implied by the criteria.
  → PASS

  Q2: What tables are 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


  Q49: Do you see problematic geographic areas, where gross sales might be inhibited by lack of sufficient payment method options?
  Narrative: **Analytical framing:** States are flagged as potentially inhibited where (a) credit card share materially exceeds the national benchmark of 78.5%, signaling low payment diversity, and (b) avg gross revenue per order falls below the national benchmark of BRL 83.05 — or where order volume is low rela...
  Assessment: The bot excelled at constructing a rigorous analytical framework with clear benchmarks, producing a well-structured state-level breakdown with multiple payment diversity dimensions, and delivering appropriately specific caveats distinguishing observed usage from platform availability and correlation from causation. The only minor gap is that the "avg gross rev/order" figures for flagged states all exceed the national benchmark substantially (BRL 145–238 vs. BRL 83), which the bot correctly attributes to freight but could have more 

## Validation Results

**50 / 50 questions passed** across all test categories (Mode 0 — Retrieve):

| Test Set | Passed | Total |
|---|---|---|
| Metadata / Descriptive | 6 | 6 |
| Easy | 10 | 10 |
| Hard | 10 | 10 |
| Misleading | 10 | 10 |
| Multi-stage | 5 | 5 |
| Realistic | 9 | 9 |
| **Total** | **50** | **50** |

Test categories cover: schema/metadata questions, simple aggregations, complex multi-join queries, edge cases designed to trigger misclassification, multi-step follow-up chains, and realistic business analyst questions. LLM-assessed reconciliation applied to all results.